In [6]:
from dotenv import load_dotenv
load_dotenv()

import json
from google import genai
from google.genai import types

with open("orders.json", "r", encoding="utf-8") as f:
    orders = json.load(f)

print(f"Loaded {len(orders)} orders")

client = genai.Client()
CHAT_MODEL = "gemini-3.1-flash-lite"

Loaded 4 orders


In [8]:
def lookup_order(order_id):
    order = orders.get(order_id)
    if order is None:
        return {"error": f"Order '{order_id}' not found."}
    return {
        "order_id": order_id,
        "item": order["item"],
        "price": order["price"],
        "purchased": order["purchased"],
        "warranty_months": order["warranty_months"]
    }


def calculate(expression):
    import re
    if not re.fullmatch(r"[0-9+\-*/().\s]+", expression):
        return {"error": f"Invalid expression: '{expression}'"}
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return {"result": result}
    except Exception as e:
        return {"error": f"Could not evaluate '{expression}': {e}"}


available_functions = {
    "lookup_order": lookup_order,
    "calculate": calculate
}

print("Tools defined.")

Tools defined.


In [9]:
lookup_order_declaration = {
    "name": "lookup_order",
    "description": "Look up an order by its order ID and return the item name, price, purchase date, and warranty length in months. Returns an error if the order ID does not exist.",
    "parameters": {
        "type": "object",
        "properties": {
            "order_id": {
                "type": "string",
                "description": "The order ID to look up, e.g. 'A1001'."
            }
        },
        "required": ["order_id"]
    }
}

calculate_declaration = {
    "name": "calculate",
    "description": "Evaluate a simple arithmetic expression (addition, subtraction, multiplication, division, parentheses) and return the numeric result. Use this for any math instead of computing it yourself.",
    "parameters": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "The arithmetic expression to evaluate, e.g. '1200 * 3'."
            }
        },
        "required": ["expression"]
    }
}

tools = types.Tool(function_declarations=[lookup_order_declaration, calculate_declaration])
config = types.GenerateContentConfig(tools=[tools])

print("Tool schemas defined.")

Tool schemas defined.


In [11]:
MAX_STEPS = 5

# Short-term memory: this list holds the entire running conversation
messages = []

def add_user_message(text):
    messages.append(types.Content(role="user", parts=[types.Part(text=text)]))

def run_agent_step():
    """
    One full turn of the agent loop for the CURRENT user message already in `messages`.
    Calls the model, handles any tool calls (looping), and returns the final text answer.
    """
    for step in range(1, MAX_STEPS + 1):
        print(f"--- Step {step} ---")

        response = client.models.generate_content(
            model=CHAT_MODEL,
            contents=messages,   # send the WHOLE running conversation every time = memory
            config=config
        )

        candidate = response.candidates[0]
        parts = candidate.content.parts
        function_calls = [p.function_call for p in parts if p.function_call is not None]

        if not function_calls:
            # Final answer — no tool requested
            final_text = response.text
            messages.append(candidate.content)  # store the model's final reply in memory too
            print(f"Model gave final answer: {final_text}")
            return final_text

        # Model requested one or more tool calls — append its request to memory
        messages.append(candidate.content)

        function_response_parts = []
        for fc in function_calls:
            func_name = fc.name
            func_args = dict(fc.args)
            print(f"Model requested tool: {func_name}({func_args})")

            if func_name not in available_functions:
                result = {"error": f"Unknown tool: {func_name}"}
            else:
                result = available_functions[func_name](**func_args)

            print(f"Tool result: {result}")

            function_response_parts.append(
                types.Part.from_function_response(
                    name=func_name,
                    response={"result": result}
                )
            )

        # Append the tool result(s) to memory, then loop again
        messages.append(types.Content(role="tool", parts=function_response_parts))

    print("Couldn't finish in time (hit step limit).")
    return "Couldn't finish in time."

print("Manual loop defined.")

Manual loop defined.


In [12]:
print("=== TURN 1 ===")
add_user_message("What did order A1001 cost?")
answer1 = run_agent_step()

print("\n=== TURN 2 ===")
add_user_message("And what about three of them?")
answer2 = run_agent_step()

print("\n" + "="*60)
print("FINAL SUMMARY")
print("Turn 1 answer:", answer1)
print("Turn 2 answer:", answer2)

=== TURN 1 ===
--- Step 1 ---
Model requested tool: lookup_order({'order_id': 'A1001'})
Tool result: {'order_id': 'A1001', 'item': 'laptop', 'price': 1200, 'purchased': '2026-05-20', 'warranty_months': 12}
--- Step 2 ---
Model gave final answer: Order A1001 cost $1,200.

=== TURN 2 ===
--- Step 1 ---
Model requested tool: calculate({'expression': '1200 * 3'})
Tool result: {'result': 3600}
--- Step 2 ---
Model gave final answer: Three of them would cost $3,600.

FINAL SUMMARY
Turn 1 answer: Order A1001 cost $1,200.
Turn 2 answer: Three of them would cost $3,600.


In [13]:
# Test the step limit with an artificially low cap
messages_test = []

def add_user_message_test(text):
    messages_test.append(types.Content(role="user", parts=[types.Part(text=text)]))

def run_agent_step_test(max_steps=1):
    for step in range(1, max_steps + 1):
        print(f"--- Step {step} ---")
        response = client.models.generate_content(
            model=CHAT_MODEL,
            contents=messages_test,
            config=config
        )
        candidate = response.candidates[0]
        parts = candidate.content.parts
        function_calls = [p.function_call for p in parts if p.function_call is not None]

        if not function_calls:
            final_text = response.text
            messages_test.append(candidate.content)
            print(f"Model gave final answer: {final_text}")
            return final_text

        messages_test.append(candidate.content)
        function_response_parts = []
        for fc in function_calls:
            func_name = fc.name
            func_args = dict(fc.args)
            print(f"Model requested tool: {func_name}({func_args})")
            result = available_functions[func_name](**func_args)
            print(f"Tool result: {result}")
            function_response_parts.append(
                types.Part.from_function_response(name=func_name, response={"result": result})
            )
        messages_test.append(types.Content(role="tool", parts=function_response_parts))

    print("Couldn't finish in time (hit step limit).")
    return "Couldn't finish in time."

add_user_message_test("For order A1001, what would the total be if I bought three of them?")
result = run_agent_step_test(max_steps=1)
print("\nResult:", result)

--- Step 1 ---
Model requested tool: lookup_order({'order_id': 'A1001'})
Tool result: {'order_id': 'A1001', 'item': 'laptop', 'price': 1200, 'purchased': '2026-05-20', 'warranty_months': 12}
Couldn't finish in time (hit step limit).

Result: Couldn't finish in time.
